# Przetwarzanie danych pogodowych w Spark i HDFS

Notebook pokazuje przepływ danych od plików JSON w HDFS przez transformacje w PySpark do zapisów w formatach Parquet i ORC. Na końcu odczytujemy zapisane dane, aby sprawdzić wyniki.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from utils.config import load_config

## Konfiguracja

Importujemy API PySpark oraz funkcję wczytującą ustawienia projektu. Konfiguracja zawiera adres klastra Spark i punkt dostępowy do HDFS.

In [ ]:
# Wczytaj ustawienia połączeń i ścieżek z pliku YAML.
config = load_config("config.yaml")

In [ ]:
sh_conf = config["spark_hadoop"]

# Adresy klastra Spark i HDFS pochodzą z konfiguracji.
spark_master = sh_conf["spark_master"]
hdfs_namenode = sh_conf["hdfs_namenode"]

# Katalog wejściowy oraz lokalizacje plików wynikowych w HDFS.
hdfs_input = f"{hdfs_namenode}/input/json/"

hdfs_output_v1 = f"{hdfs_namenode}/output/data.parquet"
hdfs_output_v2 = f"{hdfs_namenode}/output/data_v2.parquet"
hdfs_output_orc = f"{hdfs_namenode}/output/data_v2.orc"


## Uruchomienie sesji Spark

Z konfiguracji pobieramy adresy źródła i plików docelowych. Następnie tworzymy sesję Spark, która będzie wykonywać operacje na danych.

In [ ]:
# Inicjalizacja sesji Spark z adresem mastera z konfiguracji
spark = (
    SparkSession
    .builder
    .appName("JSON to Parquet")
    .master(spark_master)
    .getOrCreate()
)

In [ ]:
spark

## Wczytanie i wstępna inspekcja JSON

Dane wejściowe są odczytywane z katalogu HDFS. Poniższe komórki sprawdzają liczbę rekordów, strukturę schematu oraz przykładowe wiersze.

In [ ]:
%%time

# Odczytaj wszystkie pliki JSON z katalogu wejściowego HDFS.
df = spark.read.json(hdfs_input)

In [ ]:
%%time

df.count()

In [ ]:
%%time

df.printSchema()

In [ ]:
%%time

df.show()

## Spłaszczenie zagnieżdżonej struktury

Pola z obiektów JSON przenosimy do osobnych kolumn, co ułatwia dalsze analizy i zapis kolumnowy. Tablica `weather` jest rozwijana do wierszy; `explode_outer` zachowuje również rekordy, w których tablica jest pusta lub nieobecna.

In [ ]:
%%time

# Wybierz zagnieżdżone pola i nadaj im płaskie, jednoznaczne nazwy kolumn.
df_flat = df \
    .withColumn("clouds_all", F.col("clouds.all")) \
    .withColumn("coord_lat", F.col("coord.lat")) \
    .withColumn("coord_lon", F.col("coord.lon")) \
    .withColumn("main_feels_like", F.col("main.feels_like")) \
    .withColumn("main_grnd_level", F.col("main.grnd_level")) \
    .withColumn("main_humidity", F.col("main.humidity")) \
    .withColumn("main_pressure", F.col("main.pressure")) \
    .withColumn("main_sea_level", F.col("main.sea_level")) \
    .withColumn("main_temp", F.col("main.temp")) \
    .withColumn("main_temp_max", F.col("main.temp_max")) \
    .withColumn("main_temp_min", F.col("main.temp_min")) \
    .withColumn("rain_1h", F.col("rain.`1h`")) \
    .withColumn("sys_sunrise", F.col("sys.sunrise")) \
    .withColumn("sys_sunset", F.col("sys.sunset")) \
    .withColumn("wind_deg", F.col("wind.deg")) \
    .withColumn("wind_gust", F.col("wind.gust")) \
    .withColumn("wind_speed", F.col("wind.speed")) \
    .withColumn("weather_exploded", F.explode_outer("weather")) \
    .withColumn("weather_id", F.col("weather_exploded.id")) \
    .withColumn("weather_main", F.col("weather_exploded.main")) \
    .withColumn("weather_description", F.col("weather_exploded.description")) \
    .drop("clouds", "coord", "main", "rain", "sys", "wind", "weather", "weather_exploded", "base", "cod", "timezone")


In [ ]:
df_flat.printSchema()

In [ ]:
%%time

df_flat.show()

## Konwersja znaczników czasu

Czas pomiaru, wschodu i zachodu słońca zamieniamy na typ timestamp Spark. Po konwersji usuwamy surowe kolumny źródłowe, aby uniknąć dublowania tych samych informacji.

In [ ]:
%%time

# Zamień sekundy Unix na timestamp i usuń surowe kolumny czasu.
df_flat = df_flat \
    .withColumn("dt_ts", F.to_timestamp(F.col("dt"))) \
    .withColumn("sys_sunrise_ts", F.to_timestamp(F.col("sys_sunrise"))) \
    .withColumn("sys_sunset_ts", F.to_timestamp(F.col("sys_sunset"))) \
    .drop("dt", "sys_sunrise", "sys_sunset")

In [ ]:
%%time

df_flat.show()

In [ ]:
df_flat.printSchema()

## Zapis w HDFS

Zapisujemy dane jako Parquet w wersji rozproszonej, jako pojedynczą partycję Parquet oraz jako ORC. Tryb `overwrite` zastępuje wcześniejszą zawartość wskazanej lokalizacji.

In [ ]:
%%time

# Zapisz z domyślną liczbą partycji Spark.
df_flat.write.mode("overwrite").parquet(hdfs_output_v1)

In [ ]:
%%time

# Złącz partycje do jednego pliku wynikowego.
df_flat.coalesce(1).write.mode("overwrite").parquet(hdfs_output_v2)

In [ ]:
%%time

# Zapisz dane w kolumnowym formacie ORC.
df_flat.coalesce(1).write.mode("overwrite").format("orc").save(hdfs_output_orc)

## Odczyt i weryfikacja zapisów

Ponownie odczytujemy pliki Parquet i ORC, wyświetlamy przykładowe rekordy oraz schemat. Pozwala to sprawdzić, czy dane zostały zapisane i poprawnie zinterpretowane.

In [ ]:
%%time

# Wczytaj Parquet zapisany bez łączenia partycji i obejrzyj próbkę.
df_read = spark.read.parquet(hdfs_output_v1)
df_read.show(5, truncate=False)

In [ ]:
%%time

df_read.printSchema()

In [ ]:
%%time

# Sprawdź również Parquet zapisany jako jedna partycja.
df_read = spark.read.parquet(hdfs_output_v2)
df_read.show(5, truncate=False)

In [ ]:
%%time

df_read.printSchema()

In [ ]:
%%time

# Odczytaj próbkę z pliku ORC.
df_read = spark.read.orc(hdfs_output_orc)
df_read.show(5, truncate=False)

In [ ]:
%%time

df_read.printSchema()

## Zakończenie pracy

Po zakończeniu odczytów zamykamy sesję Spark, aby zwolnić zasoby klastra.

In [ ]:
# Zatrzymaj sesję Spark po zakończeniu pracy.
spark.stop()